In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "groq:openai/gpt-oss-120b"
)

In [2]:
from pydantic import BaseModel, Field
class movie(BaseModel):
    title: str = Field( description="The title of the movie")
    year: int = Field(descriptinon="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")
    cast: str = Field(description="famous cast of the movie")


C:\Users\DELL\AppData\Local\Temp\ipykernel_716\658984023.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptinon'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  year: int = Field(descriptinon="The year the movie was released")


In [3]:
model_with_structure = model.with_structured_output(movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B558E37F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B55922E3D0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'pa

In [4]:
model.invoke("Provide detail about the movie bahubali")

AIMessage(content='**Baahubali – Overview & Key Details**\n\n| Item | Information |\n|------|--------------|\n| **Title** | *Baahubali: The Beginning* (2015)  <br> *Baahubali: The Conclusion* (2017) |\n| **Director / Writer** | S.\u202fS.\u202fRajamouli |\n| **Producers** | Shobu Yarlagadda, Prasad V.\u202fPotluri (Arka Media Works) |\n| **Language** | Primarily Telugu (original), dubbed into Tamil, Hindi, Malayalam, and several other languages |\n| **Genre** | Epic historical‑fantasy, action‑adventure |\n| **Runtime** | *The Beginning*: 159\u202fmin (Telugu) <br> *The Conclusion*: 171\u202fmin (Telugu) |\n| **Budget** | Approx.\u202f₹1.8\u202fbillion (US\u202f$25\u202fM) for the two‑film saga – one of the most expensive Indian productions at the time |\n| **Box‑Office Gross** | Combined worldwide gross ≈\u202f₹2,500\u202fcrore (US\u202f$330\u202fM) – the highest‑grossing Indian franchise until it was surpassed by *RRR* (2022) |\n| **Awards** | 10 National Film Awards (incl. Best Featu

In [5]:
response = model_with_structure.invoke("Provide detail about the movie bahubali")
response

movie(title='Baahubali: The Beginning', year=2015, director='S. S. Rajamouli', rating=8.5, cast='Prabhas, Rana Daggubati, Anushka Shetty, Tamannaah, Sathyaraj, Ramya Krishna')

## Nested Structure

In [6]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str
class movie_details(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget : float | None = Field(None, description="budget in millions USD")
model_with_structure = model.with_structured_output(movie_details)
model_with_structure

response = model_with_structure.invoke("Provide detaabouvils about the movie bahubali")
response


movie_details(title='Baahubali: The Beginning', year=2015, cast=[Actor(name='Prabhas', role='Mahendra Baahubali / Sivudu'), Actor(name='Rana Daggubati', role='Bhallaladeva'), Actor(name='Anushka Shetty', role='Devasena'), Actor(name='Tamannaah', role='Avanthika'), Actor(name='Ramya Krishna', role='Sivagami'), Actor(name='Sathyaraj', role='Kattappa'), Actor(name='Nassar', role='Bijjaladeva'), Actor(name='Ajay', role='Shivudu (young)'), Actor(name='Alia Bhatt', role='Mahalakshmi (cameo)')], genres=['Action', 'Drama', 'Fantasy', 'Epic'], budget=18000000.0)

## TypedDict

In [12]:
from typing_extensions import TypedDict, Annotated
class movie_details(TypedDict):
    """ A movie with details"""
    title:Annotated[str,..., "The title of the movie"]
    actors:Annotated[str,...,"Famous Actors in this movie"]
    year:Annotated[int,...,"The Year the movie was released"]
    director:Annotated[str,...,"The director of movie"]
    rating: Annotated[float,..., "The movie rating out of 10"]
model_with_typed_dict= model.with_structured_output(movie_details)
model_with_typed_dict.invoke("provide detail about movie PK")

{'actors': 'Aamir Khan, Anushka Sharma, Sushant Singh Rajput, Sanjay Dutt, Pankaj Tripathi',
 'director': 'Rajkumar Hirani',
 'rating': 8.1,
 'title': 'PK',
 'year': 2014}

In [13]:
class Actor(TypedDict):
    name:str
    role:str
class movie_details(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget : float | None = Field(None, description="budget in millions USD")
model_with_structure = model.with_structured_output(movie_details)
model_with_structure

response = model_with_structure.invoke("Provide detaabouvils about the movie bahubali")
response

{'budget': 27000000,
 'cast': [{'name': 'Prabhas', 'role': 'Shivudu / Mahendra Baahubali'},
  {'name': 'Rana Daggubati', 'role': 'Bhallaladeva'},
  {'name': 'Anushka Shetty', 'role': 'Devasena'},
  {'name': 'Tamannaah', 'role': 'Avanthika'},
  {'name': 'Sathyaraj', 'role': 'Kattappa'},
  {'name': 'Ramya Krishna', 'role': 'Sivagami'}],
 'genres': ['Action', 'Drama', 'Fantasy'],
 'title': 'Baahubali: The Beginning',
 'year': 2015}

In [14]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data Class

In [16]:
from langchain.agents import create_agent
class ContactInfo(TypedDict):
    """Contact Information for a person"""
    name:Annotated[str,...,"name of person"]
    email:Annotated[str,..., "email address of the person"]
    phone:Annotated[str,...,"phone number of person"]

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [17]:
from dataclasses import dataclass
@dataclass
class CotactInfo:
    """Contact information for a person"""
    name:str
    email:str
    phone:str

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}